# Análisis de Sensibilidad: KDE vs gridsize

Objetivo: encontrar el **primer** valor de `gridsize` (entre 10,000 y 100,000)
donde el delta máximo respecto a la referencia (gridsize=100,000) sea ≤ 0.001.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from scipy.stats import gaussian_kde

## 1. Carga de datos

In [ ]:
FILE_PATH = r"/home/manuel_gonzalez/Descargas/Clustering_Microbiota/Datos/otu_data_converted.csv"

path = Path(FILE_PATH)
if not path.exists():
    raise FileNotFoundError(f"No existe el archivo: {path}")

df = pd.read_csv(path, delimiter=",")
print(f"Shape del DataFrame: {df.shape}")
df.head(3)

## 2. Preparación de datos (misma lógica que el bloque KDE original)

In [ ]:
# Parámetros KDE (iguales al bloque original)
BW_METHOD  = "scott"
UMBRAL     = 0.001   # delta máximo aceptable
GRIDSIZE_REF  = 100_000
GRIDSIZE_MIN  = 10_000
GRIDSIZE_STEP = 1_000

# Preparar datos numéricos positivos
df_num = df.select_dtypes(include=[np.number])
if df_num.empty:
    raise ValueError("El DataFrame no tiene columnas numéricas.")

valores = df_num.to_numpy(dtype=float).ravel()
valores = valores[np.isfinite(valores)]
data_kde = valores[valores > 0]

if len(data_kde) == 0:
    raise ValueError("No hay datos positivos para KDE.")

print(f"N positivos: {len(data_kde):,}")
print(f"min = {data_kde.min():.4g}   max = {data_kde.max():.4g}")

## 3. Análisis de sensibilidad

Para cada `gridsize` N:
1. Evalúa la KDE en N puntos log-espaciados en el rango de datos.
2. Interpola esos N valores a la malla de referencia (100,000 puntos).
3. Calcula el delta = `max |y_N_interpolado − y_ref|`.

In [ ]:
# Construir el objeto KDE (único: mismos datos y bw para todos los gridsizes)
kde = gaussian_kde(data_kde, bw_method=BW_METHOD)

# Malla de referencia: log-espaciada (igual al modo log_scale de seaborn)
x_ref = np.logspace(
    np.log10(data_kde.min()),
    np.log10(data_kde.max()),
    GRIDSIZE_REF
)
y_ref = kde(x_ref)  # densidad de referencia (100,000 puntos)

# Recorrer gridsizes de 10,000 a 99,000
gridsizes = np.arange(GRIDSIZE_MIN, GRIDSIZE_REF, GRIDSIZE_STEP)

deltas = []
print(f"Evaluando {len(gridsizes)} gridsizes ({GRIDSIZE_MIN:,} a {GRIDSIZE_REF-GRIDSIZE_STEP:,}, paso {GRIDSIZE_STEP:,})...")

for n in gridsizes:
    x_n = np.logspace(np.log10(data_kde.min()), np.log10(data_kde.max()), n)
    y_n = kde(x_n)
    # Interpolar a la malla de referencia
    y_interp = np.interp(x_ref, x_n, y_n)
    delta = float(np.max(np.abs(y_interp - y_ref)))
    deltas.append(delta)

deltas = np.array(deltas)

# Encontrar el primer gridsize donde delta <= UMBRAL
mask_ok = deltas <= UMBRAL
if mask_ok.any():
    idx_opt = int(np.argmax(mask_ok))  # primer True
    gridsize_opt = int(gridsizes[idx_opt])
    delta_opt   = float(deltas[idx_opt])
    print(f"\n✅ Primer gridsize con delta ≤ {UMBRAL}: {gridsize_opt:,}  (delta = {delta_opt:.6f})")
else:
    gridsize_opt = None
    print(f"\n⚠ Ningún gridsize en el rango alcanza delta ≤ {UMBRAL}.")
    print(f"  Delta mínimo encontrado: {deltas.min():.6f} en gridsize={int(gridsizes[np.argmin(deltas)]):,}")

## 4. Visualización

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Sensibilidad de KDE al parámetro gridsize\n"
             f"(bw_method='{BW_METHOD}', referencia={GRIDSIZE_REF:,}, umbral δ={UMBRAL})",
             fontsize=14)

# ── Gráfico 1: Delta vs gridsize (escala lineal) ─────────────────────────────
ax = axes[0, 0]
ax.plot(gridsizes, deltas, linewidth=1.5, color="steelblue")
ax.axhline(UMBRAL, color="tomato", linestyle="--", linewidth=1.5, label=f"Umbral δ={UMBRAL}")
if gridsize_opt is not None:
    ax.axvline(gridsize_opt, color="green", linestyle=":", linewidth=1.5,
               label=f"Óptimo: {gridsize_opt:,}")
    ax.scatter([gridsize_opt], [delta_opt], color="green", zorder=5, s=60)
ax.set_xlabel("gridsize")
ax.set_ylabel("Delta máximo")
ax.set_title("Delta máximo vs gridsize (lineal)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── Gráfico 2: Delta vs gridsize (escala log en Y) ───────────────────────────
ax = axes[0, 1]
ax.semilogy(gridsizes, deltas, linewidth=1.5, color="steelblue")
ax.axhline(UMBRAL, color="tomato", linestyle="--", linewidth=1.5, label=f"Umbral δ={UMBRAL}")
if gridsize_opt is not None:
    ax.axvline(gridsize_opt, color="green", linestyle=":", linewidth=1.5,
               label=f"Óptimo: {gridsize_opt:,}")
    ax.scatter([gridsize_opt], [delta_opt], color="green", zorder=5, s=60)
ax.set_xlabel("gridsize")
ax.set_ylabel("Delta máximo (log)")
ax.set_title("Delta máximo vs gridsize (log Y)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── Gráfico 3: KDE referencia vs KDE óptima ──────────────────────────────────
ax = axes[1, 0]
ax.plot(x_ref, y_ref, color="steelblue", linewidth=2, label=f"Referencia (gridsize={GRIDSIZE_REF:,})", zorder=3)
if gridsize_opt is not None:
    x_opt = np.logspace(np.log10(data_kde.min()), np.log10(data_kde.max()), gridsize_opt)
    y_opt = kde(x_opt)
    ax.plot(x_opt, y_opt, color="tomato", linewidth=1.5, linestyle="--",
            label=f"Óptimo (gridsize={gridsize_opt:,})", zorder=4)
ax.set_xscale("log")
ax.set_xlabel("Valor (escala log)")
ax.set_ylabel("Densidad")
ax.set_title("Curva KDE: referencia vs óptimo")
ax.legend()
ax.grid(True, alpha=0.3, which="both")

# ── Gráfico 4: Diferencia puntual (referencia − óptimo) en la malla de ref ──
ax = axes[1, 1]
if gridsize_opt is not None:
    y_opt_interp = np.interp(x_ref, x_opt, y_opt)
    diff = y_opt_interp - y_ref
    ax.plot(x_ref, diff, color="darkorange", linewidth=1)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
    ax.axhline( UMBRAL, color="tomato", linestyle="--", linewidth=1, label=f"+umbral ({UMBRAL})")
    ax.axhline(-UMBRAL, color="tomato", linestyle="--", linewidth=1, label=f"−umbral ({UMBRAL})")
    ax.set_xscale("log")
    ax.set_xlabel("Valor (escala log)")
    ax.set_ylabel("Diferencia")
    ax.set_title(f"Diferencia puntual: KDE óptimo − referencia\n(gridsize óptimo = {gridsize_opt:,})")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")
else:
    ax.text(0.5, 0.5, "No se encontró gridsize óptimo",
            ha="center", va="center", transform=ax.transAxes, fontsize=12)
    ax.set_title("Diferencia puntual")

plt.tight_layout()
plt.savefig("kde_gridsize_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada como 'kde_gridsize_sensitivity.png'")

## 5. Resumen

In [ ]:
print("="*55)
print("RESUMEN DEL ANÁLISIS DE SENSIBILIDAD KDE")
print("="*55)
print(f"  bw_method        : {BW_METHOD}")
print(f"  Datos (N pos)    : {len(data_kde):,}")
print(f"  Referencia       : gridsize = {GRIDSIZE_REF:,}")
print(f"  Rango explorado  : {GRIDSIZE_MIN:,} → {int(gridsizes[-1]):,} (paso {GRIDSIZE_STEP:,})")
print(f"  Umbral δ         : {UMBRAL}")
print("-"*55)
print(f"  Delta máximo en gridsize={GRIDSIZE_MIN:,}  : {deltas[0]:.6f}")
print(f"  Delta mínimo encontrado  : {deltas.min():.6f} (gridsize={int(gridsizes[np.argmin(deltas)]):,})")
if gridsize_opt is not None:
    print(f"  ✅ Primer gridsize con δ≤{UMBRAL}: {gridsize_opt:,}  (delta={delta_opt:.6f})")
    print(f"  Ahorro vs referencia: {GRIDSIZE_REF - gridsize_opt:,} puntos menos")
    print(f"  ({100*(GRIDSIZE_REF-gridsize_opt)/GRIDSIZE_REF:.1f}% menos que la referencia)")
else:
    print(f"  ⚠ Ningún gridsize en el rango alcanzó δ ≤ {UMBRAL}.")
print("="*55)